# CommandGemma v1 Training

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# Configuration
MODEL_NAME = "google/gemma-3-1b-it"
DATASET_NAME = "TGoddessana/common-dev-commands-linux-en-ko"
OUTPUT_DIR = "./commandgemma-adapters"
NUM_EPOCHS = 5
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
MAX_LENGTH = 256
LORA_R = 32
LORA_ALPHA = 64
TEST_SPLIT = 0.1

In [ ]:
# Load dataset
print(f"Loading dataset: {DATASET_NAME}")
raw_dataset = load_dataset(DATASET_NAME, split="train")
print(f"Loaded {len(raw_dataset)} samples")

In [ ]:
# Prepare dataset
SYSTEM_PROMPT = "You are a shell command generator. Output only the exact command, nothing else."

def create_chat_messages(question: str, command: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": command}
    ]

processed_data = []
for item in raw_dataset:
    korean_q = item.get("korean_question", "")
    english_q = item.get("english_question", "")
    command = item.get("command", "")

    if not command:
        continue

    if korean_q:
        processed_data.append({"messages": create_chat_messages(korean_q, command)})
    if english_q:
        processed_data.append({"messages": create_chat_messages(english_q, command)})

dataset = Dataset.from_list(processed_data)
dataset_splits = dataset.train_test_split(test_size=TEST_SPLIT, shuffle=True, seed=42)
print(f"Train: {len(dataset_splits['train'])}, Test: {len(dataset_splits['test'])}")
print(f"Sample: {processed_data[0]}")

In [ ]:
# Load model with 4-bit quantization
print(f"Loading model: {MODEL_NAME}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# LoRA config
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Training config
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    max_length=MAX_LENGTH,
    gradient_checkpointing=True,
    packing=False,
    optim="adamw_torch",
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=False,
    bf16=True,
)

In [ ]:
# Train
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_splits["train"],
    eval_dataset=dataset_splits["test"],
    peft_config=lora_config,
)

trainer.train()

In [ ]:
from peft import PeftModel

trainer.push_to_hub("TGoddessana/commandgemma-v1-adapters")
tokenizer.push_to_hub("TGoddessana/commandgemma-v1-adapters")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = peft_model.merge_and_unload()
merged_model.push_to_hub("TGoddessana/commandgemma-v1")
tokenizer.push_to_hub("TGoddessana/commandgemma-v1")

In [ ]:
# Test inference
test_questions = [
    "현재 디렉토리의 파일 목록을 보여줘",
    "Show me all running docker containers",
    "nginx 서비스 재시작해줘",
    "현재 디렉토리에 있는 파일 모두 zip 압축하려면 어떻게 해?",
    "현재 있는 디렉토리의 모든 파일들 크기순으로 정렬해줘"
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    outputs = model.generate(inputs, max_new_tokens=64, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {response}\n")